# V7 Train-only 데이터 진단과 모델 선택

## tl;dr

- Train 3,000행, 입력 피처 16개이며 ID 중복은 없습니다.
- target은 0~1 범위의 0.01 격자 101개 값입니다.
- CatBoost·LightGBM·XGBoost·스플라인·KNN·RandomForest는 V6보다 나빴습니다.
- 독립적인 2차원 피처쌍 1-NN 보조 모델을 15% 혼합한 V7은 고정 후 Audit3에서 V6보다 평균 MAE가 0.000374 낮았고 Seed 3/3을 이겼습니다.
- Test는 이 노트북에서 읽지 않습니다.

## Context & Methods

목표는 대회 규정을 지키면서 MAE가 낮은 후보를 찾는 것입니다.

### Key Assumptions

- 모델과 전처리 선택은 Train-only Fold 예측으로 수행합니다.
- 동일 입력 중복 행은 같은 Fold에 둡니다.
- Test 통계·행 수·인덱스·다른 Test 행 정보는 사용하지 않습니다.
- 첫 화면 Seed에서 후보를 고른 뒤 남은 Development Seed와 새 Audit3 Seed에서는 설정을 변경하지 않습니다.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

DATA_DIR = Path(os.environ["HACKATHON_DATA_DIR"])
RESULTS_DIR = Path(os.environ["HACKATHON_RESULTS_DIR"])
train = pd.read_csv(DATA_DIR / "train.csv")
TARGET = "stress_score"
ID_COLUMN = "ID"
print({"rows": len(train), "columns": len(train.columns), "input_features": len(train.columns) - 2})

{'rows': 3000, 'columns': 18, 'input_features': 16}


## Data

In [2]:
features = train.drop(columns=[TARGET, ID_COLUMN])
quality = pd.DataFrame({
    "value": [
        train[ID_COLUMN].duplicated().sum(),
        features.duplicated(keep=False).sum(),
        train[TARGET].nunique(),
        train[TARGET].min(),
        train[TARGET].max(),
        np.isclose(train[TARGET] * 100, np.round(train[TARGET] * 100)).mean(),
    ]
}, index=[
    "duplicate_id_rows",
    "duplicate_feature_rows",
    "target_unique_values",
    "target_min",
    "target_max",
    "target_on_0.01_grid_rate",
])
quality

,value
duplicate_id_rows,0.0
duplicate_feature_rows,12.0
target_unique_values,101.0
target_min,0.0
target_max,1.0
target_on_0.01_grid_rate,1.0


In [3]:
missing = features.isna().sum().sort_values(ascending=False)
pd.DataFrame({"missing_count": missing[missing > 0], "missing_rate": missing[missing > 0] / len(train)})

,missing_count,missing_rate
family_medical_history,1486,0.495333
medical_history,1289,0.429667
mean_working,1032,0.344000
edu_level,607,0.202333


## Results

In [4]:
boosting = pd.read_csv(RESULTS_DIR / "v7_boosting_screen" / "v7_family_screen_seed42_summary.csv")
local = pd.read_csv(RESULTS_DIR / "v7_local_screen" / "v7_family_screen_seed42_summary.csv")
pair = pd.read_csv(RESULTS_DIR / "v7_pair_neighbor_seed42_summary.csv")

screen = pd.DataFrame([
    {"family": "Boosting best", **boosting.iloc[0][["candidate", "mean_mae"]].to_dict()},
    {"family": "Local baseline best", **local.iloc[0][["candidate", "mean_mae"]].to_dict()},
    {"family": "Pair-neighbor best", **pair.iloc[0][["candidate", "mean_mae"]].to_dict()},
    {"family": "V6 reference on Seed 42", "candidate": "V6", "mean_mae": 0.15128333333333335},
])
screen.sort_values("mean_mae")

,family,candidate,mean_mae
3,V6 reference on Seed 42,V6,0.151283
2,Pair-neighbor best,pair_v6_8_k1_q50_raw,0.155808
1,Local baseline best,rf_q52_mf70_round2,0.164723
0,Boosting best,lgb_l1_l63_round2,0.199790


In [5]:
audit3 = pd.read_csv(RESULTS_DIR / "v7_pair_blend_audit3_summary.csv")
audit3[[
    "v6_mean_mae",
    "pair_mean_mae",
    "blend_mean_mae",
    "mean_delta_vs_v6",
    "seed_win_rate",
    "paired_ci95_lower",
    "paired_ci95_upper",
]]

,v6_mean_mae,pair_mean_mae,blend_mean_mae,mean_delta_vs_v6,seed_win_rate,paired_ci95_lower,paired_ci95_upper
0,0.147019,0.151524,0.146644,-0.000374,1.0,-0.000791,0.000044


## Takeaways

1. 이 데이터에서는 평활화가 강한 부스팅·선형 스플라인보다 세밀한 국소 이웃 구조가 유리했습니다.
2. Pair-neighbor 단독은 V6보다 약했지만 오차가 달라 15% 혼합 시 Development 확인 Seed와 Audit3 Seed를 모두 이겼습니다.
3. Audit3 신뢰구간 상한은 0.000044로 0을 아주 조금 포함합니다. 따라서 개선 가능성은 일관되지만 Public 성능을 보장할 수는 없습니다.
4. 이상치는 Train에서 오류로 확인되지 않았으므로 임의 삭제하지 않았습니다.
5. 최종 코드는 Test 각 행을 Train 기준 경험적 순위에만 투영하므로 Test 전체 통계를 사용하지 않습니다.